# Atualiza_Status_Cameras_CCTV_v4: testar as câmeras e gravar o status (agora também a palavra "online"/"offline")

⚠ **Altera dados** da camada *Cameras_Apucarana*. Faz **backup** antes, em `Documents\Backup_CCTV`.

**Novidade da v4 (pedido do Ricardo):** cria e preenche os campos **`txt_online`** e **`txt_offline`**, com a palavra que o app mostra ao lado do bairro na lista, em verde ou vermelho.

Continua fazendo tudo da v3:
- grava `status`, `status_cam`, `status_video`, `tempo_resposta`, `ultima_verificacao`, `lista_online` e `lista_offline` (bolinhas);
- **`url_player`**: vídeo real se a câmera estiver online, página "Câmera offline" se estiver fora do ar;
- **`cctv`** continua guardando o endereço original.

A saída (.txt) vai para a pasta **Saidas** do projeto aberto no Pro.
**Run All.** Depois rode o **Cria_Cones_Visao** para atualizar a cor dos cones.
**Esta passa a ser a versão oficial do verificador** (pode apagar a v3).

In [ ]:
# =====================================================================
# CÉLULA 1 de 3 - LER A CAMADA E FAZER BACKUP
# =====================================================================
import os, json, time, csv, builtins, warnings
from datetime import datetime
warnings.filterwarnings("ignore")
NOME = "Atualiza_Status_Cameras_CCTV_v4"
try:
    import arcpy
    _RAIZ = arcpy.mp.ArcGISProject("CURRENT").homeFolder
except Exception:
    _RAIZ = os.path.join(os.path.expanduser("~"), "Documents")
_SAIDA = next((os.path.join(_RAIZ, n) for n in ("Saidas", "Saida") if os.path.isdir(os.path.join(_RAIZ, n))), os.path.join(_RAIZ, "Saidas"))
os.makedirs(_SAIDA, exist_ok=True)
LOG_TXT = os.path.join(_SAIDA, NOME + ".txt"); _LOG = open(LOG_TXT, "w", encoding="utf-8")
_p = builtins.print
def print(*a, **k):
    _p(*a, **k); k.pop("file", None); _p(*a, **k, file=_LOG); _LOG.flush()
print("▶ Célula 1 iniciada...")
import requests, urllib3
urllib3.disable_warnings()
from arcgis.gis import GIS
from arcgis.features import FeatureLayer
gis = GIS("pro")
URL = "https://geo.apucarana.pr.gov.br/server/rest/services/Hosted/Cameras_Apucarana/FeatureServer/0"
fl = FeatureLayer(URL, gis=gis)
P = fl.properties
OID = P.objectIdField
CAMPOS = {f["name"]: f for f in P.fields}
# v4: campos da palavra "online"/"offline" mostrada na lista
novos = [{"name": n, "type": "esriFieldTypeString", "alias": a, "length": 10, "nullable": True, "editable": True}
         for n, a in (("txt_online", "Palavra online (lista)"), ("txt_offline", "Palavra offline (lista)")) if n not in CAMPOS]
if novos:
    print(f"   Criando campos {[f['name'] for f in novos]}: {fl.manager.add_to_definition({'fields': novos})}")
    fl = FeatureLayer(URL, gis=gis); P = fl.properties; CAMPOS = {f["name"]: f for f in P.fields}
    if not all(n in CAMPOS for n in ("txt_online", "txt_offline")):
        raise RuntimeError("PAREI: não consegui criar os campos txt_online/txt_offline (precisa ser dono da camada).")
print(f"   Camada: {P.name} · OID={OID} · edição: {P.get('capabilities')}")
for c in ("status", "status_cam", "status_video", "tempo_resposta", "ultima_verificacao", "lista_online", "lista_offline", "txt_online", "txt_offline"):
    f = CAMPOS.get(c)
    print(f"   campo {c}: {f['type'].replace('esriFieldType','') + ' tam ' + str(f.get('length','-')) if f else 'NÃO EXISTE'}")
fs = fl.query(where="1=1", out_fields="*", return_geometry=False)
REG = [f.attributes for f in fs.features]
BKP_DIR = os.path.join(os.path.expanduser("~"), "Documents", "Backup_CCTV"); os.makedirs(BKP_DIR, exist_ok=True)
BKP = os.path.join(BKP_DIR, f"Cameras_Apucarana_{datetime.now():%Y%m%d_%H%M%S}.json")
json.dump(REG, open(BKP, "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=str)
print(f"   {len(REG)} câmeras · backup: {BKP}")
# aprender o padrão já usado em lista_online / lista_offline
def padrao(st):
    for r in REG:
        if str(r.get("status")).lower() == st: return (r.get("lista_online"), r.get("lista_offline"))
    return None
PAD_ON = ("●", None)      # v3: pontinho verde na lista (online)
PAD_OFF = (None, "●")     # v3: pontinho vermelho na lista (offline)
print(f"   Padrão lista_online/lista_offline → online: {PAD_ON} · offline: {PAD_OFF}")
OFFLINE_URL = "https://jrnegrao.github.io/cctv/offline.html"   # trocar para o servidor da Prefeitura na versão oficial
from urllib.parse import quote
f_url = CAMPOS.get("url_player")
print(f"   Página offline: {OFFLINE_URL} · campo url_player: tamanho {f_url.get('length') if f_url else 'NÃO EXISTE'}")
if f_url is None or "cctv" not in CAMPOS: raise RuntimeError("PAREI: a camada precisa dos campos url_player e cctv.")
# garantir que 'cctv' tem o endereço original do vídeo
FONTE = {}
for r in REG:
    c, u = (r.get("cctv") or "").strip(), (r.get("url_player") or "").strip()
    if c and "offline.html" not in c: FONTE[r[OID]] = c
    elif u and "offline.html" not in u: FONTE[r[OID]] = u
    else: FONTE[r[OID]] = ""
sem = [r.get("descricao") for r in REG if not FONTE[r[OID]]]
print(f"   Endereço original do vídeo encontrado em {len(REG) - len(sem)}/{len(REG)} câmeras {('· SEM endereço: ' + str(sem)) if sem else ''}")
print("✔ Célula 1 concluída com sucesso.")


In [ ]:
# =====================================================================
# CÉLULA 2 de 3 - TESTAR OS VÍDEOS E GRAVAR NA CAMADA
# =====================================================================
print("▶ Célula 2 iniciada: testando os vídeos...")
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/130 Safari/537.36"}
def cabe(campo, valor):
    f = CAMPOS.get(campo)
    if f is None: return None
    if isinstance(valor, str) and f.get("length"): return valor[: f["length"]]
    return valor
agora_ms = int(time.time() * 1000)
UPD, MUD = [], []
for r in REG:
    u = FONTE[r[OID]]
    ok, seg = False, None
    if u:
        try:
            t0 = time.time(); resp = requests.get(u.rstrip("/") + "/index.m3u8", headers=UA, timeout=15, verify=False)
            seg = time.time() - t0; ok = resp.status_code == 200
        except Exception:
            ok = False
    novo = {OID: r[OID]}
    vals = {"status": "Online" if ok else "Offline",
            "status_cam": "ONLINE" if ok else "OFFLINE",
            "status_video": "AO VIVO" if ok else "SEM VIDEO",
            "tempo_resposta": f"{seg:.2f} s" if (ok and seg is not None) else None,
            "ultima_verificacao": agora_ms,
            "lista_online": PAD_ON[0] if ok else PAD_OFF[0],
            "lista_offline": PAD_ON[1] if ok else PAD_OFF[1],
            "txt_online": "online" if ok else None,
            "txt_offline": None if ok else "offline",
            "cctv": u or None}
    agora_txt = datetime.fromtimestamp(agora_ms / 1000).strftime("%d/%m/%Y %H:%M")
    url_off = f"{OFFLINE_URL}?nome={quote(str(r.get('descricao') or ''))}&verif={quote(agora_txt)}"
    if f_url.get("length") and len(url_off) > f_url["length"]: url_off = OFFLINE_URL
    vals["url_player"] = u if ok else url_off
    for k, v in vals.items():
        if k in CAMPOS: novo[k] = cabe(k, v)
    UPD.append({"attributes": novo})
    if str(r.get("status")) != vals["status"]: MUD.append(f"{r.get('descricao')}: {r.get('status')} → {vals['status']}")
    print(f"   {'🟢' if ok else '🔴'} {r.get('descricao')} · {vals['status']}" + (f" · {seg:.2f} s" if ok else ""))
print(f"\n   Mudanças de status: {MUD or 'nenhuma'}")
res = fl.edit_features(updates=UPD)
falhas = [x for x in res.get("updateResults", []) if not x.get("success")]
print(f"   Gravação: {len(res.get('updateResults', [])) - len(falhas)} ok · {len(falhas)} falhas {falhas if falhas else ''}")
print("✔ Célula 2 concluída com sucesso.")


In [ ]:
# =====================================================================
# CÉLULA 3 de 3 - CONFERIR
# =====================================================================
print("▶ Célula 3 iniciada: conferindo...")
fs2 = fl.query(where="1=1", out_fields="descricao,status,status_video,tempo_resposta,ultima_verificacao,url_player", return_geometry=False, order_by_fields="descricao")
on = 0
for f in fs2.features:
    a = f.attributes; on += a.get("status") == "Online"
    vv = datetime.fromtimestamp(a["ultima_verificacao"]/1000).strftime("%d/%m/%Y %H:%M") if a.get("ultima_verificacao") else "-"
    print(f"   {a.get('descricao'):40} {a.get('status'):8} {str(a.get('status_video')):13} {str(a.get('tempo_resposta') or ''):8} verificado {vv}  {'(página offline)' if 'offline.html' in str(a.get('url_player')) else ''}")
print(f"\n   Resultado: {on} online · {len(fs2.features) - on} offline")
print("")
print("✔ Célula 3 concluída com sucesso.")
print("Links:")
print("   App (Portal): https://geo.apucarana.pr.gov.br/portal/apps/experiencebuilder/experience/?id=a7640a4685224019aceed2b795930f9f")
print("   App local (mosaico): https://localhost:3001/experience/2/")
print(f"   Camada: {URL}")
print(f"   Backup: {BKP}")
print(f"   Registro: {LOG_TXT}")
